# AttB sites occupancy
Breseq output's isn't very clear. Also, it doesn't really show the circular phage.

In [2]:
from pathlib import Path
from collections import Counter
from Bio import SeqIO, Seq

illumina_folder = Path('../../../datasets/genomics/illumina_reads/')

In [3]:
# Downstream sequences
yciI_down = 'TGGTTTCACTGAAACGTGTTCATAG'
glgS_down = 'AGACTTTCTCTTCCAGAACTTCTCT'
yeeJ_down = 'GAGTGTTCCATTCGGGGCACTCTTT'
phag_down = 'TGTCATTTGGCATATTACGAACAAT'

sites_top = [yciI_down, glgS_down, yeeJ_down, phag_down]
sites_bot = [str(Seq.reverse_complement(i)) for i in sites_top]
sites_names = ['yciI', 'glgS', 'yeeJ', 'phag']

In [ ]:
len(attB), len(phag_down)

## Extract the reads

In [64]:
def lowercase_low_qual_bases(seq, qual, thr = 32):
    return ''.join([b.lower() if q < thr else b for b,q in zip(seq, qual)])

In [85]:
folders = sorted([i for i in illumina_folder.iterdir() if len(i.name) == 4])

In [91]:
output_file.close()

# Load file

In [253]:
import pandas as pd
import numpy as np

matching_reads_all = pd.read_csv('../../../datasets/genomics/09_phage_fhuA/attB_matching_reads.tsv', sep = '\t')

attB = 'agaacactttcttaaat'
attB_rev = 'ATTTAAGAAAGTGTTCT'

phage_end = 'TTTAAGTGCAAATTTCTAATAAATT'
phage_end_rev = 'AATTTATTAGAAATTTGCACTTAAA'

In [254]:
bp = 30
matching_reads = matching_reads_all[matching_reads_all.upstream.str.len() >= bp].reset_index(drop = True)

# "tag": recombination site + sequence upstream of it in the read
matching_reads['tag'] = matching_reads.upstream.str[:bp].str.upper()

# Assign reads

In [286]:
from io import StringIO

tag_to_status = pd.read_csv(StringIO(
'''site, tag, status
yciI, TTTAAGAAAGTGTTCTGAATAGAGATTCAA, B
yciI, TTTAAGAAAGTGTTCTAATTTATTAGAAAT, P
yeeJ, TTTAAGAAATTATAACGGAAAATCACAGGT, B
yeeJ, TTTAAGAAAGTGTTCTAATTTATTAGAAAT, P
glgS, TTTAATATGGTTTTTTATTTTTCACTAAAA, B
glgS, TTTAAGAAAGTGTTCTAATTTATTAGAAAT, P
phag, ATTTAAGAAAGTGTTCTGAATAGAGATTCA, yciI
phag, ATTTAAGAAATTATAACGGAAAATCACAGG, yeeJ
phag, ATTTAATATGGTTTTTTATTTTTCACTAAA, glgS
phag, ATTTAAGAAAGTGTTCTAATTTATTAGAAA, circ
'''
), sep = ', ')

tag_to_status

merged = matching_reads.merge(tag_to_status, on = ['site', 'tag'], how = 'left').dropna()

# Remove upstream junctions with the phage
merged = merged[(merged.site != 'phag') | (merged.status == 'circ')]

/tmp/ipykernel_703368/1027676773.py:3: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  tag_to_status = pd.read_csv(StringIO(


In [294]:
counts = merged.groupby(['name', 'site', 'status']).size().reset_index(name = 'n')
counts.to_csv('../../../datasets/genomics/09_phage_fhuA/junctions_counts.tsv', sep = '\t', index = False)